# ADMM-based XAS spectrum

Per GMD bin we solve

$$ \mathrm{Cov}(A,A)\, x = \mathrm{Cov}(A, G) $$

with the regularised ADMM solver in `admm_ghost.py`. `A` is the (background-subtracted) VLS spectrum and `G` is the per-shot GMD. The resulting `x` has the same shape as one VLS spectrum; **the XAS estimate is `1/x`**.

This notebook is intended for aggregates built with `GROUP_BY_ENERGY = False`, where every open-shutter section across the energy scan contributes to a single combined Cov(A,A) / Cov(A,G) pair per GMD bin. It will still work with `GROUP_BY_ENERGY = True` by picking a single energy bin via `ENERGY_BIN`.

ADMM penalty weights almost certainly need re-tuning per dataset.

In [ ]:
import sys
from pathlib import Path

import numpy as np
import matplotlib.pyplot as plt

%matplotlib inline

cwd = Path.cwd().resolve()
repo_root = None
for p in [cwd] + list(cwd.parents):
    if (p / "analysis" / "scripts").exists():
        repo_root = p
        break
if repo_root is None:
    raise RuntimeError("Could not locate repository root containing analysis/scripts")
sys.path.insert(0, str(repo_root / "analysis" / "scripts"))

import config as path_config
from compute_aggregates import load_aggregates
from admm_ghost import solve_admm

## Load aggregates

Point `AGG_PATH` at an aggregates H5 written by `compute_xas_aggregates.py`. Setting `GROUP_BY_ENERGY = False` in the source config gives a single energy bin (`ENERGY_BIN = 0` below).

In [ ]:
RUN_NO = 58780
AGG_PATH = Path(path_config.COMBINED_DIR) / f"run{RUN_NO}_xas_aggregates.h5"

agg = load_aggregates(AGG_PATH)
if agg.mode != "xas_scan":
    raise RuntimeError(f"expected mode='xas_scan', got {agg.mode!r}")

vls_pixels = np.asarray(agg.vls_pixels)
gmd_edges  = np.asarray(agg.gmd_edges)
n_pixels   = vls_pixels.size
N_E, N_GMD = agg.n_per_bin.shape

print(f"file:    {AGG_PATH}")
print(f"run:     {agg.metadata.get('run_no')}")
print(f"shape:   N_E={N_E}, N_GMD={N_GMD}, n_pixels={n_pixels}")
print(f"GMD edges: {gmd_edges}")
print(f"group_by_energy: {agg.metadata.get('group_by_energy')}")
print(f"total shots: {int(agg.n_per_bin.sum())}")

## ADMM parameters

`ENERGY_BIN` selects which energy slice to invert. With `GROUP_BY_ENERGY = False` only index 0 is meaningful; with `GROUP_BY_ENERGY = True` choose by index into `agg.nominal_energies`.

The smoothness / sparsity weights are dataset-dependent — start with the defaults and rescale based on the residual plots.

In [ ]:
ENERGY_BIN          = 0
MIN_SHOTS_PER_BIN   = 100
LAMBDA_SMOOTH_PIXEL = 1e-2
LAMBDA_SPARSE       = 1e-4
RHO                 = 1.0
MAX_ITER            = 500
DIFF_ORDER          = 2

if not 0 <= ENERGY_BIN < N_E:
    raise IndexError(f"ENERGY_BIN={ENERGY_BIN} out of range [0, {N_E - 1}].")

e_label = agg.nominal_energies[ENERGY_BIN]
if np.isnan(e_label):
    e_label_str = "all energies"
else:
    e_label_str = f"{float(e_label):.2f} eV"
print(f"ENERGY_BIN={ENERGY_BIN}  ({e_label_str})")

## Covariance preview

Quick look at `Cov(A,A)` and `Cov(A,G)` for the highest-statistics GMD bin in `ENERGY_BIN`, before running the solver.

In [ ]:
n_per_bin = agg.n_per_bin[ENERGY_BIN]            # (N_GMD,)
preview_g = int(np.argmax(n_per_bin))

A_mean = agg.A[ENERGY_BIN, preview_g]            # (n_pixels,)
AtA    = agg.AtA[ENERGY_BIN, preview_g]          # (n_pixels, n_pixels)
G_mean = float(agg.G[ENERGY_BIN, preview_g])
AtG    = agg.AtG[ENERGY_BIN, preview_g]          # (n_pixels,)

M = AtA - np.outer(A_mean, A_mean)
b = AtG - A_mean * G_mean

fig, axes = plt.subplots(1, 2, figsize=(11, 4.4))
vlim = float(np.nanpercentile(np.abs(M), 99)) if np.isfinite(M).any() else 1.0
im0 = axes[0].imshow(
    M, aspect="auto", cmap="RdBu_r", origin="lower",
    extent=[vls_pixels[0], vls_pixels[-1], vls_pixels[0], vls_pixels[-1]],
    vmin=-vlim, vmax=vlim,
)
fig.colorbar(im0, ax=axes[0])
axes[0].set_title(f"Cov(A, A)  ({e_label_str}, GMD bin {preview_g})")
axes[0].set_xlabel("VLS pixel")
axes[0].set_ylabel("VLS pixel")

axes[1].plot(vls_pixels, b, color="tab:blue")
axes[1].axhline(0, color="k", lw=0.6, alpha=0.5)
axes[1].set_xlabel("VLS pixel")
axes[1].set_ylabel("Cov(A, G)")
axes[1].set_title(f"Cov(A, G)  (GMD bin {preview_g}, n={int(n_per_bin[preview_g])} shots)")
axes[1].grid(alpha=0.3)
fig.tight_layout()
plt.show()

## Solve per GMD bin

For each GMD bin with at least `MIN_SHOTS_PER_BIN` shots: form `M`, `B = Cov(A,G).reshape(-1, 1)`, run the ADMM solver, take `x = res.X.ravel()` and `XAS = 1/x`. Bins with insufficient statistics are skipped.

In [ ]:
x_per_bin   = np.full((N_GMD, n_pixels), np.nan, dtype=np.float64)
xas_per_bin = np.full((N_GMD, n_pixels), np.nan, dtype=np.float64)
results = {}

for g in range(N_GMD):
    n = int(agg.n_per_bin[ENERGY_BIN, g])
    if n < MIN_SHOTS_PER_BIN:
        print(f"  GMD bin {g}: n={n} < {MIN_SHOTS_PER_BIN}, skipping")
        continue

    A_mean = agg.A[ENERGY_BIN, g]
    AtA    = agg.AtA[ENERGY_BIN, g]
    G_mean = float(agg.G[ENERGY_BIN, g])
    AtG    = agg.AtG[ENERGY_BIN, g]
    if not (np.isfinite(A_mean).all() and np.isfinite(AtA).all()
            and np.isfinite(AtG).all() and np.isfinite(G_mean)):
        print(f"  GMD bin {g}: NaN in aggregates, skipping")
        continue

    M = AtA - np.outer(A_mean, A_mean)
    B = (AtG - A_mean * G_mean).reshape(-1, 1)

    res = solve_admm(
        M, B,
        lambda_smooth_pixel=LAMBDA_SMOOTH_PIXEL,
        lambda_smooth_tof=0.0,
        lambda_sparse=LAMBDA_SPARSE,
        rho=RHO,
        diff_order=DIFF_ORDER,
        max_iter=MAX_ITER,
    )
    x = res.X.ravel()
    x_per_bin[g] = x
    with np.errstate(divide="ignore", invalid="ignore"):
        xas_per_bin[g] = 1.0 / x
    results[g] = res
    print(f"  GMD bin {g}: n={n:>6d}  iter={res.n_iter:>3d}  "
          f"converged={res.converged}  "
          f"final pri={res.history['primal'][-1]:.2e}  "
          f"final dual={res.history['dual'][-1]:.2e}")

## Solver output `x`

Sanity-check the raw solution before inverting. Zeros (from L1 shrinkage) and sign changes in `x` will produce singularities / sign-flipped features in `1/x`, so inspect this first.

In [ ]:
fig, ax = plt.subplots(figsize=(9.0, 4.4))
cmap = plt.get_cmap("viridis")
for g in range(N_GMD):
    if not np.isfinite(x_per_bin[g]).any():
        continue
    label = f"[{gmd_edges[g]:.2f}, {gmd_edges[g+1]:.2f}) uJ  (n={int(agg.n_per_bin[ENERGY_BIN, g])})"
    ax.plot(vls_pixels, x_per_bin[g], color=cmap(g / max(N_GMD - 1, 1)),
            lw=1.3, label=label)
ax.axhline(0, color="k", lw=0.6, alpha=0.5)
ax.set_xlabel("VLS pixel")
ax.set_ylabel("x")
ax.set_title(f"ADMM solution x ({e_label_str})")
ax.grid(alpha=0.3)
ax.legend(fontsize=8, title="GMD bin", ncol=2)
fig.tight_layout()
plt.show()

## XAS = 1 / x per GMD bin

In [ ]:
fig, ax = plt.subplots(figsize=(9.0, 4.6))
cmap = plt.get_cmap("viridis")
for g in range(N_GMD):
    if not np.isfinite(xas_per_bin[g]).any():
        continue
    label = f"[{gmd_edges[g]:.2f}, {gmd_edges[g+1]:.2f}) uJ  (n={int(agg.n_per_bin[ENERGY_BIN, g])})"
    ax.plot(vls_pixels, xas_per_bin[g], color=cmap(g / max(N_GMD - 1, 1)),
            lw=1.4, label=label)
ax.set_xlabel("VLS pixel")
ax.set_ylabel("XAS = 1 / x")
ax.set_title(f"ADMM-derived XAS per GMD bin ({e_label_str})")
ax.grid(alpha=0.3)
ax.legend(fontsize=8, title="GMD bin", ncol=2)
fig.tight_layout()
plt.show()

## Convergence diagnostics

In [ ]:
if not results:
    print("No converged bins to plot.")
else:
    fig, axes = plt.subplots(1, 2, figsize=(11, 3.8), sharex=True)
    cmap = plt.get_cmap("viridis")
    for g, res in results.items():
        color = cmap(g / max(N_GMD - 1, 1))
        label = f"bin {g}"
        axes[0].semilogy(res.history["primal"], color=color, label=label)
        axes[1].semilogy(res.history["dual"],   color=color, label=label)
    axes[0].set_title("primal residual")
    axes[1].set_title("dual residual")
    for a in axes:
        a.set_xlabel("iteration")
        a.grid(alpha=0.3)
    axes[0].legend(fontsize=8, ncol=2)
    fig.tight_layout()
    plt.show()